# Apply masks and clean up labkit segmented images

In [ ]:
from pathlib import Path

from bioio import BioImage
import bioio_ome_tiff
import bioio_tifffile
from bioio.writers import OmeTiffWriter
import bioio_tifffile

import numpy as np
import pandas as pd

%matplotlib notebook
%matplotlib inline
import matplotlib.pyplot as plt

import sys
src_path = str(Path.cwd().parent.parent)
if src_path not in sys.path:
    sys.path.append(src_path)
import microscopy_analysis.d00_utils.utilities as utils
import microscopy_analysis.d00_utils.dirnames as dn
from microscopy_analysis.d01_init_proc import vis_and_rescale as vr
from microscopy_analysis.d01_init_proc import subtractbg
from microscopy_analysis.d01_init_proc import applymask as am

from scipy import ndimage as ndi
from skimage.measure import label, regionprops

from skimage.filters import threshold_otsu, threshold_multiotsu
from skimage import morphology, segmentation
import numpy.ma as ma

from tqdm import tqdm
from PIL import Image

In [ ]:
input_dirpath = Path(input())

In [ ]:
caaxch = 0
cellch = 1
seg_caaxch = 2
seg_cellch = 3

# get mask dirpaths
proc_dirpath = utils.get_proc_dirpath(input_dirpath)
polygonmasks_dirpath = proc_dirpath / dn.masks_dirname / dn.polygon_ROI_dirname


output_dirpath = input_dirpath.parent / (input_dirpath.name + '_edited')

fig_dirpath = output_dirpath / dn.figs_dirname
fig_dirpath.mkdir(exist_ok=True, parents=True)

maskpaths = [path for path in polygonmasks_dirpath.glob('*.png')]
maskpaths.sort()

for maskpath in tqdm(maskpaths):
    
    ROI_name = maskpath.name.split('.png')[0] + '.ome.tif'
    savepath = output_dirpath / ROI_name
    imgpath = input_dirpath / ROI_name
    
    if (not savepath.is_file()) & (imgpath.is_file()):

        img_file = BioImage(imgpath, reader=bioio_ome_tiff.Reader)
        img = img_file.data
    
        # get previous segmentations
        seg_cell = (img[:, seg_cellch, np.newaxis, :, :, :] > 0).astype('bool')
        seg_caax = (img[:, seg_caaxch, np.newaxis, :, :, :] > 0).astype('bool')
    
        # get polygon masks
        polygonmask = np.array(Image.open(maskpath))
    
        # apply cellmask to cell channel
        seg_cell = (seg_cell * polygonmask)
        seg_cell = am.remove_unconnected_areas(seg_cell)
        seg_cell = am.remove_small_holes(seg_cell, area_threshold=5)
        
        # mask caax channel with cell channel
        seg_caax = am.mask_img(seg_caax, seg_cell)
        seg_caax = am.remove_small_holes(seg_caax, area_threshold=5)
        seg_caax = am.remove_small_objects(seg_caax, min_size=5)
    
        # get cmp areas from newly calculated cellmask and caax
        seg_cmp = (seg_cell & ~seg_caax)
        seg_cmp = am.remove_cmp_near_cellborders(seg_cell, seg_cmp)
    
        # recalculate seg_cell with edited cmp
        seg_cell = (seg_caax | seg_cmp)
        seg_cell = am.remove_unconnected_areas(seg_cell)

        # re-mask all channels with edited seg_cell
        seg_cmp = (seg_cmp & seg_cell)
        seg_caax = (seg_caax & seg_cell)

        img_edited = np.concatenate([img[:, [caaxch, cellch], :, :, :], seg_cmp, seg_caax], axis=1)
        img_edited = img_edited.astype(img_file.dtype)
        ome_metadata = utils.construct_ome_metadata(img_edited, img_file)
        
        ROI_name = maskpath.name.split('.png')[0] + '.ome.tif'
        OmeTiffWriter.save(img_edited, (output_dirpath / ROI_name), ome_xml=ome_metadata)

        comp_fig = vr.create_cmp_fig(seg_cmp, seg_caax)
        OmeTiffWriter.save(comp_fig, fig_dirpath / imgpath.name)

In [ ]:
# resave manually edited images
manually_edited_dirpath = Path(input())

In [ ]:
imgpaths = [path for path in manually_edited.glob('*edited.ome.tif')]
print(len(imgpaths))

for imgpath in tqdm(imgpaths):
    img = BioImage(imgpath, reader=bioio_tifffile.Reader).data   
    ome_metadata = utils.construct_ome_metadata(img, img_file)
    OmeTiffWriter.save(img, imgpath, ome_xml=ome_metadata)

In [ ]:
caax_cell_actin_seg_edited_dirpath = manually_edited_dirpath.parent
older_dirpath = caax_cell_actin_seg_edited_dirpath / 'older'
for imgpath in imgpaths:
    prev_imgname = imgpath.name.replace("_edited", "")
    older_imgpath = caax_cell_actin_seg_edited_dirpath / prev_imgname
    
    if older_imgpath.is_file():
        older_imgpath.rename(older_dirpath / prev_imgname)

    imgpath.rename(caax_cell_actin_seg_edited_dirpath / prev_imgname)
        